In [ ]:
# import the required packages
from ingest import load_faq_data
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex
import numpy as np
from tqdm.auto import tqdm
from dotenv import load_dotenv
import os

In [ ]:
# load the environment variables
load_dotenv()

hf_token = os.getenv('HF_TOKEN')

# specify the persistent storage paths
DB_DIR='.'
DB_NAME='faq.db'

# load the embedding model
model = SentenceTransformer(model_name_or_path='all-MiniLM-L6-v2', token=hf_token)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# load the documents
documents = load_faq_data()

texts = []
vectors = []
batch_size = 50

# create a corpus of 'question answer' from documents
for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

# embed the text corpus in batches and conslidate all the embeddings in an embedding matrix
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

In [ ]:
# index the document vectors
vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path=os.path.join(DB_DIR, DB_NAME)
)

vs_index.fit(X, documents)
vs_index.close()